# Visualize Outpainting Bounding Boxes
So sánh ảnh gốc vs ảnh outpainted với bounding box

In [ ]:
import xml.etree.ElementTree as ET
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import os

# === CONFIG ===
IMAGE_ID = "hd02"  # <-- Change this to view other images

ORIG_IMG = f"/users/PGS0407/binben14/VietHuy/construction-site/SODA/data/SODA VOCdevkit/VOCdevkit/VOC2007/JPEGImages/{IMAGE_ID}.jpg"
ORIG_XML = f"/users/PGS0407/binben14/VietHuy/construction-site/SODA/data/SODA VOCdevkit/VOCdevkit/VOC2007/Annotations/{IMAGE_ID}.xml"
OUT_IMG  = f"/users/PGS0407/binben14/VietHuy/construction-site/augmentation_data/SODA/small/JPEGImages/{IMAGE_ID}.jpg"
OUT_XML  = f"/users/PGS0407/binben14/VietHuy/construction-site/augmentation_data/SODA/small/Annotations/{IMAGE_ID}.xml"

# Colors per class
COLORS = {
    "person": "red", "helmet": "lime", "vest": "cyan",
    "hook": "yellow", "fence": "magenta", "crane": "orange",
    "machinery": "blue", "vehicle": "pink",
}
DEFAULT_COLOR = "white"

In [ ]:
def parse_voc_xml(xml_path):
    """Parse VOC XML and return list of (name, xmin, ymin, xmax, ymax)."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find("size")
    w, h = int(size.findtext("width")), int(size.findtext("height"))
    objects = []
    for obj in root.findall("object"):
        bb = obj.find("bndbox")
        objects.append((
            obj.findtext("name"),
            int(float(bb.findtext("xmin"))),
            int(float(bb.findtext("ymin"))),
            int(float(bb.findtext("xmax"))),
            int(float(bb.findtext("ymax"))),
        ))
    return w, h, objects


def draw_boxes(img, objects, line_width=3):
    """Draw bounding boxes on a copy of the image."""
    img = img.copy()
    draw = ImageDraw.Draw(img)
    for name, x1, y1, x2, y2 in objects:
        color = COLORS.get(name, DEFAULT_COLOR)
        draw.rectangle([x1, y1, x2, y2], outline=color, width=line_width)
        # Label
        draw.text((x1, max(0, y1 - 15)), name, fill=color)
    return img

In [ ]:
# Load original
orig_img = Image.open(ORIG_IMG)
orig_w, orig_h, orig_objs = parse_voc_xml(ORIG_XML)
orig_drawn = draw_boxes(orig_img, orig_objs, line_width=4)

# Load outpainted
out_img = Image.open(OUT_IMG)
out_w, out_h, out_objs = parse_voc_xml(OUT_XML)
out_drawn = draw_boxes(out_img, out_objs, line_width=4)

print(f"Original:   {orig_img.size} | {len(orig_objs)} objects")
print(f"Outpainted: {out_img.size} | {len(out_objs)} objects")

In [ ]:
# Side by side comparison
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

axes[0].imshow(orig_drawn)
axes[0].set_title(f"Original ({orig_w}x{orig_h}) - {len(orig_objs)} objects", fontsize=14)
axes[0].axis("off")

axes[1].imshow(out_drawn)
axes[1].set_title(f"Outpainted ({out_w}x{out_h}) - {len(out_objs)} objects", fontsize=14)
axes[1].axis("off")

plt.suptitle(f"Image: {IMAGE_ID}", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Zoomed outpainted image with boxes
fig, ax = plt.subplots(1, 1, figsize=(16, 12))
ax.imshow(out_drawn)
ax.set_title(f"Outpainted {IMAGE_ID} - {out_w}x{out_h}", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Print bbox details
print("=== Original Bboxes ===")
for name, x1, y1, x2, y2 in orig_objs:
    print(f"  {name:12s}  ({x1:4d}, {y1:4d}, {x2:4d}, {y2:4d})  size=({x2-x1}x{y2-y1})")

print(f"\n=== Outpainted Bboxes ===")
for name, x1, y1, x2, y2 in out_objs:
    print(f"  {name:12s}  ({x1:4d}, {y1:4d}, {x2:4d}, {y2:4d})  size=({x2-x1}x{y2-y1})")

In [ ]:
# === Batch: view multiple outpainted samples ===
out_dir = "/users/PGS0407/binben14/VietHuy/construction-site/augmentation_data/SODA/small"
samples = sorted(os.listdir(os.path.join(out_dir, "JPEGImages")))[:6]

fig, axes = plt.subplots(2, 3, figsize=(24, 16))
for ax, fname in zip(axes.flat, samples):
    img_id = os.path.splitext(fname)[0]
    img = Image.open(os.path.join(out_dir, "JPEGImages", fname))
    xml_path = os.path.join(out_dir, "Annotations", f"{img_id}.xml")
    if os.path.exists(xml_path):
        _, _, objs = parse_voc_xml(xml_path)
        img = draw_boxes(img, objs, line_width=3)
    else:
        objs = []
    ax.imshow(img)
    ax.set_title(f"{img_id} ({len(objs)} obj)", fontsize=12)
    ax.axis("off")

plt.suptitle("Outpainted SODA Samples", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()